# Part 7 — The artifact atlas and the rules of record

Every claim in this project resolves to a file. This part is the map from claim to file,
and the set of rules that decides which file counts.

Two questions it answers. *Where does this number come from?* — start at the claim, follow
it to its source path. *What breaks if I change this?* — start at the artifact, read its
consumers.

![Artifact lifecycle, one swimlane per storage location. Generated by `tools/make_process_diagrams.py`.](figures/fig_artifact_atlas.png)

## 1. One home per artifact

The rule that makes verification a finite job: every artifact has exactly one producer,
one home, and a known set of consumers. When a number is challenged there is never a
question of which copy is authoritative.

- **Claim → file.** `RESEARCH.md` §8 names the source file; the table's *Lives at* column
  locates it; *Produced by* names the code that would have to be rerun to reproduce it.
- **File → blast radius.** *Consumed by* bounds what a change invalidates. A fix to the
  export code invalidates conversion bundles and everything downstream, and leaves the
  ROC arrays untouched.

### Reading the table

Paths are repository-relative. `<round>` is a campaign such as `r14`, `<leaf>` is a run
identity `<config>-s<seed>`, `<hash>` is the config hash a conversion run is keyed by, and
`<N>` in `rf<N>` is the reuse factor (RF).

Rows that say *cluster* or *mulder* live off the laptop. Everything else is in the
research working tree.

| Artifact | Produced by | Lives at | Consumed by |
|---|---|---|---|
| Experiment config JSON | `configs/gen_r14.py` (one generator per round) | `bnjettag/code/hgq2/configs/*.json` | preflight, training pods, `roc_final.py`, `convert_final.py` |
| Job manifest + launch script | `gen_r14_jobs.py` | `bnjettag/code/jobs/training/variants/kai-bn*-*.yaml`, `launch_r*.sh` | `kubectl`, from the laptop |
| Code ConfigMap | `make_code_configmap_r14.sh` | cluster object `kai-bn14-code`, one frozen ConfigMap per round | training pods (untarred to `/work/code`) |
| In-pod training outputs | `bnhgq2/train.py` | pod `emptyDir` — evaporates with the pod | the durability upload, nothing else |
| Checkpoint of record | durability gate in `bnhgq2/train.py` | W&B artifact `model-<leaf>` | `roc_final.py`, `convert_final.py`, `ebops_r14.py` |
| ROC score arrays | `roc_final.py eval-all` (cluster job) | `bnjettag/roc-results/<round>/[n<N>/]<VARIANT>-s<seed>.npz` | AUC recomputation, uncertainty pass, ROC figures |
| ROC tables + overlay | `roc_final.py` | `roc_auc.md`, `roc_overlay.png`, beside the arrays | `RESEARCH.md` tables, after verification |
| Checkpoint cache | `roc_final.py` (fetch step) | `bnjettag/roc-results/<round>/[n<N>/]_ckpt_dl/<leaf>/` | `ebops_r14.py`, local conversion |
| EBOPs table | `ebops_r14.py` | `bnjettag/results/r14/ebops_r14.json` | Pareto figures, W&B run summaries |
| Conversion bundle | `convert_final.py` | `bnjettag/results/final/runs/<hash>/<leaf>/` (and the per-round stores) | mulder (the tarball), gate audits (the JSONs) |
| Synthesis workspace | `mulder_csynth.sh` | mulder, `~/bnjet_r14/` for round 14 | `fetch_mulder_reports.sh` |
| Fetched synthesis reports | `fetch_mulder_reports.sh` | the owning run's store: raw `csynth.xml` beside parsed `csynth_report.json` | hardware tables, `log_hls_wandb.py` |
| Hardware table of record | written from parsed reports, then independently re-derived | `bnjettag/results/r14/hls_r14.md` | `RESEARCH.md`, reports, poster |
| Figures | generators in `bnjettag/code/plots/` | `docs/reports/figures-final/`, `docs/reports/figures-slides/`, `bnjettag/results/r14/figures/` | reports, slides, this book |
| Living record | updated only after gates pass | `RESEARCH.md` (§8 maps every claim to its source file) | the PI; every part of this book |
| Frozen reports | one per deliverable | `docs/reports/`, indexed by `docs/reports/INDEX.md` | corrected by dated notes, never rewritten |

### Three conventions run through the table

- **Generators outrank their outputs.** A config JSON or job manifest is never edited by
  hand: the generator is the pre-registration record, and a hand edit detaches the
  artifact from it silently.
- **Ephemerality is explicit.** The only artifacts allowed to vanish (the pod's
  `emptyDir`, the mulder workspace) are exactly those with a mandatory egress step — the
  W&B upload and the fetch script. Each egress failure mode is fatal or loud.
- **Raw stays beside parsed.** Every `csynth_report.json` sits next to the `csynth.xml`
  it came from, so the parse itself stays checkable.

### Namespaces are firewalls, not folders

`bnjettag/roc-results/` holds one directory per campaign, round 14 further split by
sequence length (`n8` through `n64`). Each store documents its own boundary:
`bnjettag/roc-results/final/README.md` declares itself definitive for its campaign and
warns against mixing it with the neighbouring round-5 arrays. An array from the wrong
directory is not a slightly different number; it answers a different question on a
different test set.

Two stores sit at the margins. Local checkpoint copies under `bnjettag/models/` and the
`_ckpt_dl/` caches are working conveniences, kept out of version control — the versioned
W&B artifact is the only checkpoint of record. Figures are fully in the record but have
three homes by audience, unified by the rule that each is emitted by a named generator
and each caption states it.

## 2. One checkpoint, end to end

The round-14 model that reached silicon is W1A8, N = 8, seed 3 — era 2, the
three-feature L1-realistic inputs. Its whole life is on disk.

1. **Config.** `bnjettag/code/hgq2/configs/r14-l1x3-n8-w1a8.json`, one of twenty round-14
   configs emitted by `configs/gen_r14.py`.
2. **Code ConfigMap.** `make_code_configmap_r14.sh` packs the code tree into the cluster
   object `kai-bn14-code`; it refuses to publish a tree that does not hold exactly twenty
   `r14-*.json` configs.
3. **Job.** `kai-bn14-l1x3-n8-w1a8-s3.yaml`, generated by `gen_r14_jobs.py` and launched
   by `launch_r14.sh`, binds config to seed to GPU. The pod fetches the dataset from
   Zenodo, trains, and writes `model_best.keras`, `train_meta.json` and `input_std.json`
   into its own `emptyDir`.
4. **Checkpoint of record.** Those three files exist nowhere today, because the pod is
   gone. They survive as the versioned W&B artifact `model-r14-l1x3-n8-w1a8-s3`
   (project `BNJetTagAug`, group `r14-n8`), pushed before the pod was allowed to exit.
5. **Scores.** ROC job `kai-bn14-roc-n8.yaml` fetches that artifact, applies the
   checkpoint's own `input_std.json`, scores the 260,000 held-out jets, and writes
   `bnjettag/roc-results/r14/n8/W1A8-s3.npz` — three keys, `y`, `score`, `meta`, with
   `--expect-n 260000` guaranteeing the count. **ROC-test macro-OvR AUC 0.8724**
   (n = 260,000), recomputed from the arrays (`bnjettag/roc-results/r14/n8/roc_auc.md`);
   all sixty round-14 arrays were re-verified on 2026-08-03.
6. **Conversion and gates.** `convert_final.py` rebuilds the export graph into
   `bnjettag/results/final/runs/38a20c62/w1a8-s3-r14n8/`, keyed by the config hash
   `38a20c62`: GATE 1 score correlation **0.998816** against the trained model, GATE 2
   bit-exact C-simulation (both gates defined in part 6), plus `convert.json`, `ebops.json`, `hls_prj_rf1/` and the
   payload `hls_prj_rf1.tar.gz`.
7. **Synthesis.** The tarball goes by `scp` to mulder's `~/bnjet_r14/`, where
   `mulder_csynth.sh` runs Vitis HLS and `parse_csynth.py` turns `csynth.xml` into
   `csynth_report.json` on the spot. `fetch_mulder_reports.sh` brings both files home into
   `csynth_rf1/` and, for the second operating point, `csynth_rf1_fabric/`.
8. **Row of record.** The numbers enter `bnjettag/results/r14/hls_r14.md` (2026-08-07)
   only after an independent pass re-derived every cell from the raw XML
   (`bnjettag/results/r14/verification_hls_r14_2026-08-07.txt`).

### Step 6, read from the bundle

`export_verify.json` is the gate verdict, not a summary of one. It carries the threshold
and the policy alongside the measured correlation, so the pass is auditable years later.

In [ ]:
import json

v = json.load(open("examples/export_verify_r14n8_w1a8s3.json"))

print("config        ", v["config"], " hash", v["config_hash"])
print("checkpoint    ", v["checkpoint"])
print("GATE 1 corr   ", round(v["gate1"]["corr_scores"], 6),
      " threshold", v["gate1_threshold"],
      " pass:", v["gate1_pass"])
print("beta encoding ", v["gate1"]["beta_mode"], " on", v["gate1"]["n"], "jets")

The same file carries the training monitor. The two AUCs live one key apart and are never
interchangeable: the monitor picks the epoch, the ROC-test number is the one reported.

In [ ]:
print("validation AUC (early-stopping monitor, macro-OvR):",
      round(v["train_meta"]["best_val_macro_auc"], 4))
print("ROC-test AUC  (number of record, macro-OvR, n = 260,000): 0.8724")
print("   recomputed from bnjettag/roc-results/r14/n8/W1A8-s3.npz")

The bundle also preserves what it replaced. The round-14 export first failed GATE 1 on
the norm-free graph and was repaired by the v5 in-graph β restoration; the superseded
verdicts stay in the same directory as `*.pre-v5-2026-08-04.json`. That episode is in
part 8.

In [ ]:
# runs on: mulder — the tarball is the only thing that travels
!./mulder_csynth.sh hls_prj_rf1.tar.gz

# The fetch script regenerates csynth_report.json from the real csynth.xml on mulder
# before copying either file home, so a report that merely rode along inside the
# tarball can never be mistaken for a result.

In [ ]:
reports = [("fx8            ", "examples/csynth_report_r14n8_fx8_rf1.json"),
           ("fabric-multiply", "examples/csynth_report_r14n8_fabric_rf1.json")]

for name, path in reports:
    r = json.load(open(path))
    lut = 100 * r["LUT"] / r["avail"]["LUT"]
    dsp = 100 * r["DSP"] / r["avail"]["DSP"]
    print(f"{name}  RF = 1, Latency strategy, part {r['part']}")
    print(f"    LUT {r['LUT']:>9,} ({lut:5.1f}% of device)"
          f"   DSP {r['DSP']:>5,} ({dsp:4.1f}%)")
    print(f"    latency {r['LatencyBest']} cycles, II = {r['IntervalMin']},"
          f" target clock {r['target_clock_ns']} ns")

### The row that ends the chain

`bnjettag/results/r14/hls_r14.md` (2026-08-07), at **RF = 1**, Latency strategy, VU13P at
a 2.5 ns target:

- **fx8:** 164 cycles, 4,133 DSPs (33.6%), 3,178,720 LUTs (184.0%) — every binary matmul
  at zero DSPs.
- **fabric-multiply:** 0 DSPs across all 106 instances, 4,376,222 LUTs (253.3%),
  146 cycles, initiation interval 1 — the first whole-model synthesis of the stack with no
  DSPs anywhere.

The fabric point is quoted with its qualification: its estimated clock carries 18
timing-violation flags from the synthesis tool, cycles and initiation interval unaffected,
until a post-route check settles it. `log_hls_wandb.py` then registered the raw XML as the
W&B artifacts `synthesis-r14n8_w1a8s3_fx8_rf1` and
`synthesis-r14n8_w1a8s3_fx8_rf1_fabricmul`, closing the provenance loop.

### What that adds up to

One checkpoint: a config JSON, a job manifest, a ConfigMap, three in-pod files, one model
artifact, one `.npz` with two companion files, a cached download, an EBOPs entry, six
conversion files, two mulder workspaces, two fetched report sets, a row in the table of
record, and two synthesis artifacts.

Nothing in that list is a copy of record of anything else. Each is a distinct stage of
evidence.

### W&B is the lineage graph, not the record

Per `docs/infrastructure/wandb-layout.md` the chain runs: a `dataset` reference artifact
(checksums only, n = 260,000 asserted) is used by each training run; the training run
produces `model-<leaf>`; that model is used by the `eval` run behind the `.npz` arrays and
by the `hls` run that registered the synthesis. For any model the graph answers "what
data, which run, which evaluation, which silicon" as a walk along typed edges, which is
what makes a two-month-old checkpoint auditable without archaeology.

But every terminal number lives in a local, parsed, versioned file. When mirror and record
disagree, the local file wins and the disagreement becomes a finding. The cloud remembers;
the repository decides.

## 3. The rules of record

Eight rules. None was adopted on principle alone — each is followed by the reason it
holds, and where an incident produced it, the date it was earned. The incidents
themselves are part 8.

### The era firewall

Era 1 is everything trained before 2026-07-01, on a private two-class dataset with a
ROC-test set of n = 222,912. Era 2 is everything since, on the public HLS4ML LHC Jet
five-class dataset, ROC-test n = 260,000. They measure different tasks on different data,
so no era-1 number appears beside an era-2 number: not in a sentence, not in a table, not
on a figure.

The firewall is physical rather than remembered. Everything era-1 was archived wholesale
under `archive/era1/`, so a current document cannot cite an era-1 file without the path
announcing what it is doing.

*The same logic at finer grain:* round 14 changed the inputs to three features per
constituent, so every number it produced is tagged `l1x3` and the tag travels with the
number wherever it appears beside 16-feature results (`RESEARCH.md` §5‴). Numbers are
compared input-matched, or the comparison is labelled cross-input context, or it is not
made.

### The two-AUC rule

Every accuracy figure is exactly one of two things: a **validation AUC** (the
early-stopping monitor) or a **ROC-test AUC** (the number of record, on the held-out
split, n = 260,000, after training is over). The monitor decides which epoch's weights
survive; only the ROC-test number is reported; every table labels which one it carries.

**Earned:** 2026-06-22, the project's first results day, when the difference between what
the training loop saw and what a referee should be shown had to be written down.

### W&B is provenance and durability, never the number of record

Training runs on ephemeral pods, so the durable home of every checkpoint and metric stream
is W&B — and none of those copies may be quoted. The layout document states the role in
one line (`docs/infrastructure/wandb-layout.md`).

**Why:** every AUC is recomputed locally from the stored `.npz` arrays and every resource
figure is parsed from the raw csynth XML, so the mirrors exist to make drift *detectable*,
not citable. The mechanism runs both ways: the ROC stage backfills each run's ROC-test AUC
into its W&B summary next to the validation AUC, precisely so a mismatch is visible to
inspection rather than hidden by separation.

### The claim-to-source map

`RESEARCH.md` §8 is a table titled "Where every number comes from": one row per claim,
each naming the file the claim is recomputable from — the `.npz` arrays for accuracy, the
csynth XML and parsed tables for hardware, the config generators for architecture facts —
plus the durable W&B artifact behind it. Its closing sentence is the operative rule:
**anything not traceable to the table is not a project claim.**

**Why:** the map is used in both directions. Writing flows through it (a number enters a
document only with its source path attached), and auditing flows back along it (pick a
claim, follow the path, recompute, and either reproduce the number or log a defect).

Where a metric's *convention* could drift rather than its value, the convention is checked
in too: the background-rejection implementation carries a self-test that must reproduce
all 25 previously published rejection means from the stored per-jet arrays before the
metric is used on anything new.

### Frozen reports, corrected only by dated notes

Documents delivered to the PI or recorded as milestones live in `docs/reports/` and are
frozen: corrections happen as dated notes inside the file, never as silent rewrites
(`docs/reports/INDEX.md`).

**Why:** a report that can be quietly repaired after delivery is not evidence of what was
known when.

The one deliberate exception is `slides/`, which is regenerated rather than frozen. The
status deck's builder parses its source tables at build time and asserts 30 headline cells
before a slide is drawn, so the build fails if a number has moved. A frozen document and a
generated one are both honest; a hand-edited generated one is neither, which is why the
index forbids editing the `.pptx` directly.

### Labels on every number

Three labels ride with every accuracy figure:

- the **metric** — validation or ROC-test, macro-OvR named;
- the **era**, and the input tag where it applies;
- the **status** — single-run or seed-averaged.

The era-2 baseline table is the visible case: FP32 and W8A8 rows read "single-run
baseline" while the binary rows read "seed-averaged", in a dedicated status column
(`RESEARCH.md` §5′).

Hardware numbers carry their own mandatory label, the **reuse factor**. A LUT or DSP count
without its RF is meaningless, because folding trades resources against cycles.

### Structural facts and measured facts

A **structural** fact follows from what the design is: when the QKeras path emits weights
as literal 1-bit `ap_uint<1>` types, a DSP multiplier is impossible for those operands by
construction (`RESEARCH.md` §6′). A **measured** fact is a line in a C-synthesis report —
the binary core's DSP count was first measured at zero on 2026-06-24, on the era-1-shape
model through the QKeras path (`RESEARCH.md` §7), and the whole-model zero-DSP point is a
2026-08-07 measurement at RF = 1.

A structural argument licenses an expectation, never a claim, because the toolchain can
defeat structure.

**Earned:** 2026-07-09. Under Resource strategy a binary matmul that "should" have used no
multipliers synthesized to 256 real DSPs at RF = 256, traced not to the weights but to a
narrow dense accumulator (`bnjettag/results/hgq2/constraints_map.md`). Only the
per-instance report caught it. The full story is in part 8.

### Corrections are findings

A mismatch found by verification is treated as a result: logged, fixed, and annotated with
a dated note at the place the wrong number appeared.

**Earned:** 2026-07-26. `RESEARCH.md` §2 carries a note withdrawing the claim that
norm-free 8-bit "beats full precision" — the comparison had crossed the normalization
axis, and once the matched norm-free FP32 control existed the durable reading was the
ordinary one: 8-bit is nearly free, not superior. `STATUS-2026-08-08.md` carries a
same-day note of the same kind, recording six seed standard deviations that had been
transcribed from 4-decimal per-seed values and were re-derived from the `.npz` arrays.

Neither error reached a reader, and both notes remain in place. The record shows not just
the right number but the fact that a wrong one existed and how it was caught.

---

Part 8 is the other half of this: the rounds in order, where the time actually went, and
the incidents that earned the rules above.